<a href="https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/Copy_of_w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khadija-Azam05/ML-Projects/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/Khadija-Azam05/ML-Projects.git

Cloning into 'ML-Projects'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 180 (delta 74), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 1.89 MiB | 13.99 MiB/s, done.
Resolving deltas: 100% (74/74), done.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### 1. Method choice and why

I chose a Random Forest because my lane is Refresh / Content Opportunity Scoring, where several content and search signals can work together in different ways. The signal audit showed that impressions and average position are useful directional signals, while update age was more mixed. A Random Forest can capture relationships between these signals without assuming that the relationship is simply linear. I will compare its results with the Week-4 baseline rather than assuming that a more complex model is automatically better.


In [ ]:
import pandas as pd

df = pd.read_csv("ML-Projects/data/raw/content_refresh_anonymized.csv")

print("Shape:", df.shape)

print("\nSelected modeling fields:")
print([
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "days_since_last_update"
])

print("\nTarget-related fields:")
print([
    "trend_direction",
    "trend_pct"
])

Shape: (30000, 44)

Selected modeling fields:
['impressions_90d', 'clicks_90d', 'avg_position', 'days_since_last_update']

Target-related fields:
['trend_direction', 'trend_pct']


In [ ]:
# Inspect the available trend fields before choosing the target

for col in ["trend_direction", "trend_pct"]:
    print(f"\n--- {col} ---")
    print("Missing:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())

    if df[col].dtype == "object":
        print(df[col].value_counts(dropna=False))
    else:
        print(df[col].describe())


--- trend_direction ---
Missing: 0
Unique values: 5
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

--- trend_pct ---
Missing: 3388
Unique values: 2712
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64


### Target choice

I will use **trend_direction** as the target because it provides a clear categorical outcome for the content opportunity task and has no missing values. The largest group is **down** with 16,262 pages, followed by **stable** with 5,962 and **up** with 4,388. I will exclude trend-derived fields such as **trend_pct** and **trend_direction** from the model features to avoid leakage.


In [ ]:
print("Target distribution:")
print(df["trend_direction"].value_counts())

print("\nTarget proportions:")
print(df["trend_direction"].value_counts(normalize=True).round(3))

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Target proportions:
trend_direction
down      0.542
stable    0.199
up        0.146
new       0.075
flat      0.038
Name: proportion, dtype: float64


### Target choice

I will use **trend_direction** as the target because it gives a clear category for each page and has no missing values. The target is not evenly balanced: 54.2% of the pages are **down**, while 19.9% are **stable**, 14.6% are **up**, 7.5% are **new**, and 3.8% are **flat**. Because of this imbalance, I will not rely on accuracy alone when evaluating the model. Trend-derived fields will also be kept out of the features to avoid leakage.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Split design

I will use an 80/20 stratified split so that each **trend_direction** category is represented in both the training and validation sets. Stratification is important because the target is imbalanced, with **down** making up more than half of the pages. I will use a fixed random seed so the split can be reproduced. Trend-derived fields will be excluded from the features to reduce the risk of target leakage.


In [ ]:
from sklearn.model_selection import train_test_split

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "avg_position"
]

X = df[feature_cols].copy()
y = df["trend_direction"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))
print("\nFeatures used:")
print(feature_cols)

Training rows: 24000
Validation rows: 6000

Features used:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'avg_position']


The split produced 24,000 training rows and 6,000 validation rows. The target proportions are almost identical in both sets, so stratification preserved the original class balance. I will use the validation set for the final model comparison.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train a Random Forest using the training split and use the model's probability of a page being **down** as its ranking score. I will compare this with the Week-4 baseline, which ranks pages mainly by 90-day impressions. Both rankings will be evaluated on the same validation set using NDCG@100, with **down** treated as the relevant outcome. This keeps the comparison focused on whether the model improves the ranking of pages that show declining movement rather than assuming that a more complex model is automatically better.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import ndcg_score
import pandas as pd

# Target: pages with a "down" trend
y_train_down = (y_train == "down").astype(int)
y_valid_down = (y_valid == "down").astype(int)

# Random Forest
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

# Train model
model.fit(X_train, y_train_down)

# Model ranking score:
# probability that a page is "down"
model_score = model.predict_proba(X_valid)[:, 1]

# Week-4 baseline:
# use impressions from the ORIGINAL dataframe
# using exactly the same validation row indices
baseline_score = (
    df.loc[X_valid.index, "impressions_90d"]
      .fillna(0)
      .to_numpy()
)

# Pages with "down" trend are treated as relevant
relevance = y_valid_down.to_numpy().reshape(1, -1)

# Calculate NDCG@100
model_ndcg = ndcg_score(
    relevance,
    model_score.reshape(1, -1),
    k=100
)

baseline_ndcg = ndcg_score(
    relevance,
    baseline_score.reshape(1, -1),
    k=100
)

# Comparison table
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "metric": [
        "NDCG@100",
        "NDCG@100"
    ],
    "score": [
        baseline_ndcg,
        model_ndcg
    ]
})

print(comparison.to_string(index=False))

print(
    "\nModel improvement over baseline:",
    round(model_ndcg - baseline_ndcg, 4)
)

         method   metric    score
Week-4 baseline NDCG@100 0.475782
  Random Forest NDCG@100 0.926767

Model improvement over baseline: 0.451


### Result

The Random Forest achieved an NDCG@100 of 0.9268, compared with 0.4758 for the Week-4 baseline. On the same validation set, the model improved the ranking score by 0.4510. This suggests that the model was better than the simple impressions-based baseline at ranking pages with a **down** trend near the top. This is a measured validation result, not evidence of causation or of how any search engine algorithm works.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# Inspect model predictions and the types of mistakes it makes

pred_down = (model.predict(X_valid) == 1)

error_check = pd.DataFrame({
    "actual": y_valid_down.to_numpy(),
    "predicted": pred_down.astype(int),
    "model_score": model_score
})

# Confusion matrix counts
print("Prediction counts:")
print(pd.crosstab(
    error_check["actual"],
    error_check["predicted"],
    rownames=["Actual"],
    colnames=["Predicted"]
))

# Show the strongest model scores
print("\nTop 10 model-ranked pages:")
print(
    error_check
    .sort_values("model_score", ascending=False)
    .head(10)
    .to_string(index=False)
)

# Error rate
errors = (error_check["actual"] != error_check["predicted"]).sum()
total = len(error_check)

print("\nValidation errors:", errors)
print("Validation rows:", total)
print("Error rate:", round(errors / total, 3))

Prediction counts:
Predicted     0     1
Actual               
0          1522  1226
1           854  2398

Top 10 model-ranked pages:
 actual  predicted  model_score
      1          1     0.837379
      1          1     0.833118
      1          1     0.821581
      1          1     0.819069
      1          1     0.817654
      1          1     0.816862
      1          1     0.816422
      1          1     0.815550
      1          1     0.814538
      0          1     0.814080

Validation errors: 2080
Validation rows: 6000
Error rate: 0.347


The model correctly identified 2,398 of the 3,252 pages that were actually moving down, but it missed 854 of them. It also marked 1,226 pages as down when their actual trend was different, showing that the ranking still contains false positives. The top-ranked pages were mostly down, although some non-down pages received high scores, so the model should be treated as a decision-support ranking rather than a final automatic refresh decision.

In [ ]:
print("Actual down pages:", y_valid_down.sum())
print("Correctly identified down pages:", ((y_valid_down == 1) & (pred_down)).sum())
print("Missed down pages:", ((y_valid_down == 1) & (~pred_down)).sum())
print("False positives:", ((y_valid_down == 0) & (pred_down)).sum())

Actual down pages: 3252
Correctly identified down pages: 2398
Missed down pages: 854
False positives: 1226
